# clipsmith — Cloud (Azure ACI)

Trigger a full cloud run from Colab: clipsmith provisions Azure resources, runs the
pipeline inside Docker, uploads clips to Google Drive, and tears everything down.
Colab just orchestrates — all heavy compute happens in Azure.

**What this notebook does:**
1. Installs clipsmith with cloud dependencies
2. Loads credentials from Colab Secrets
3. Writes `config.yaml` (pointing at your pre-built Docker image)
4. Runs `clipsmith cloud run` — provisions ACI, polls until done, uploads to Drive, tears down
5. Prints the Google Drive link to your clips

**Requirements (add all to Colab Secrets 🔑):**
| Secret | Where to get it |
|---|---|
| `AZURE_SUBSCRIPTION_ID` | Portal → Subscriptions |
| `AZURE_CLIENT_ID` | Service Principal (see docs/cloud.md) |
| `AZURE_CLIENT_SECRET` | Service Principal |
| `AZURE_TENANT_ID` | Service Principal |
| `ANTHROPIC_API_KEY` | console.anthropic.com |
| `TWITCH_CLIENT_ID` | dev.twitch.tv/console |
| `TWITCH_CLIENT_SECRET` | dev.twitch.tv/console |
| `DOCKER_HUB_USERNAME` | hub.docker.com |
| `DOCKER_HUB_PASSWORD` | Docker Hub read-only access token |
| `GOOGLE_DRIVE_FOLDER_ID` | Folder ID from Drive URL |

**Pre-requisite:** Build and push the Docker image locally before running this notebook:
```powershell
clipsmith cloud build
```

> **Cost:** ~$0.30 per 2-hr VOD (4 vCPU ACI, ~60 min runtime). Resources are deleted automatically.

In [ ]:
# @title 1. Install clipsmith with cloud dependencies
!pip install -q "clipsmith-ai[cloud]"

In [ ]:
# @title 2. Load credentials from Colab Secrets
import os
from google.colab import userdata

required = [
    "AZURE_SUBSCRIPTION_ID",
    "AZURE_CLIENT_ID",
    "AZURE_CLIENT_SECRET",
    "AZURE_TENANT_ID",
    "ANTHROPIC_API_KEY",
    "TWITCH_CLIENT_ID",
    "TWITCH_CLIENT_SECRET",
    "DOCKER_HUB_USERNAME",
    "DOCKER_HUB_PASSWORD",
    "GOOGLE_DRIVE_FOLDER_ID",
]

missing = []
for key in required:
    val = userdata.get(key) or ""
    os.environ[key] = val
    if not val:
        missing.append(key)

if missing:
    raise RuntimeError(f"Missing required secrets: {', '.join(missing)}")

print("All credentials loaded.")

In [ ]:
# @title 3. Configure — set your VOD ID and Docker image here
from pathlib import Path

VOD_ID = "2758856582"  # <-- paste your Twitch VOD ID
GAME = "Just Chatting"  # <-- game name (used as Drive subfolder)
DATE = ""  # <-- YYYY-MM-DD or leave blank for today
DOCKER_IMAGE = "ricardogr007/clipsmith:latest"  # <-- your Docker Hub image

config_yaml = f"""\
cloud:
  location: eastus
  aci_cpu: 4.0
  aci_memory_gb: 16.0
  docker_image: "{DOCKER_IMAGE}"
  gpu_sku: ""

transcribe:
  model: medium
  compute_type: int8
  language: es

llm:
  provider: anthropic
  model_anthropic: claude-haiku-4-5-20251001

clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none

caption:
  enabled: false
"""

Path("config.yaml").write_text(config_yaml)
print(f"Config written — VOD: {VOD_ID}, game: {GAME}, image: {DOCKER_IMAGE}")

In [ ]:
# @title 4. Validate Azure credentials before launching
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "clipsmith", "cloud", "setup", "--config", "config.yaml"],
    check=True,
)

In [ ]:
# @title 5. Run — provision ACI, process VOD, upload to Drive, tear down
# This cell runs for ~60–80 min. The kernel must stay alive (don't close the tab).
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "clipsmith",
    "cloud",
    "run",
    VOD_ID,
    "--game",
    GAME,
    "--config",
    "config.yaml",
    "-v",
]
if DATE:
    cmd += ["--date", DATE]

subprocess.run(cmd, check=True)

## Done

Clips are in your Google Drive under `<root_folder>/<game>/<date>/`.
The Drive link is printed at the end of cell 5.
Azure resources have been deleted — you are no longer being billed.